In [8]:
#codigo comentado vem de versao antiga do transformers
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [6]:
# qea = pipeline('text-generation', model = 'pierreguillou/bert-base-cased-squad-v1.1-portuguese')

In [7]:
# texto = 'Carl Sagan foi um cientista norte-americano. Sagan é autor de mais de 600 publicações científicas e também de mais de vinte livros de ciência e ficção científica.'
# pergunta = 'Quantas publicações científicas Carl sagan tem publicado?'
# resposta = qea(question=pergunta, context=texto)
# print('Pergunta: ', pergunta)
# print('Resposta: ', resposta['answer'])
# print('Score: ', resposta['score'])

In [18]:
#carrega tokenizador e modelo diretamente
nome_modelo = 'pierreguillou/bert-base-cased-squad-v1.1-portuguese'
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
model = AutoModelForQuestionAnswering.from_pretrained(nome_modelo)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
#texto e pergunta usados
texto = 'Carl Sagan foi um cientista norte-americano. Sagan é autor de mais de 600 publicações científicas e também de mais de vinte livros de ciência e ficção científica.'
pergunta = 'Quantas publicações científicas Carl sagan tem publicado?'

In [26]:
#tokeniza a pergunta e o texto juntos
inputs = tokenizer(pergunta, texto, add_special_tokens=True, return_tensors = 'pt')
input_ids = inputs['input_ids'].tolist()[0]
inputs

{'input_ids': tensor([[  101,   807,  6474,  8695, 12577,  7108, 14419, 22285,   376,  3565,
           136,   102,  7108,  9065,   190,   262,   222, 10989,  1410,   118,
          3725,   119,  9065,   190,   253,  1368,   125,   325,   125, 10081,
          8695, 12577,   122,   407,   125,   325,   125,  4698,  2978,   125,
          5206,   122,  6893,  5330,   119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [21]:
#passa os dados pelo modelo para obter as previsoes (logits)
with torch.no_grad():
    outputs = model(**inputs)

In [22]:
#encontra os indices com os maiores scores para o inicio e fim da resposta
answer_start_scores = outputs.start_logits
answer_end_scores = outputs.end_logits

In [23]:
answer_start = torch.argmax(answer_start_scores)
answer_end = torch.argmax(answer_end_scores) + 1

In [24]:
#converte os tokens de volta para a string legivel
resposta = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end])
)

In [25]:
#calcula um score aproximado de confianca (usando probabilidade maxima obtida)
score_confianca = float(torch.max(torch.softmax(answer_start_scores, dim=-1)))

print('Pergunta: ', pergunta)
print('Resposta: ', resposta)
print('Score: ', round(score_confianca, 4))

Pergunta:  Quantas publicações científicas Carl sagan tem publicado?
Resposta:  mais de 600
Score:  0.7328
